# Grain-structure geometrification for 2D conformal meshing

Tracked as `confMesh2d_geometrify.ipynb`.

**Role:** polygonise a Monte-Carlo grain structure, smooth grain boundaries, then hand the polygons to `gsmesh2d.mesh_gs`.

This is not the canonical mesher demo. For mesh / plot / Abaqus INP only, use `confMesh2d_gmsh.ipynb`.

Requires `gmsh` (`pip install upxo[mesh]`). `confMesh2d` (pygmsh) is deprecated.

## 1. Generate and characterise an MCGS slice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from upxo.ggrowth.mcgs import mcgs
from upxo.pxtal.geometrification import polygonised_grain_structure
from upxo.meshing.gsmesh2d import mesh_gs, visualize_gs_mesh
from upxo.meshing.writer_ABQ import summarize_inp
from pathlib import Path
_XLS = next(
    (p / "src" / "upxo" / "demos" / "confMesh" / "confMesh1.xls"
     for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "upxo" / "demos" / "confMesh" / "confMesh1.xls").is_file()),
    Path("confMesh1.xls"),
)
print("dashboard:", _XLS)

In [ ]:
pxt = mcgs(input_dashboard=str(_XLS))
pxt.simulate()
pxt.detect_grains(library="cc3d", connectivity=4)
gstslice = pxt.gs[list(pxt.gs.keys())[-1]]
gstslice.char_morph_2d(
    bbox=True, use_version=2, bbox_ex=True, area=True, eq_diameter=True,
    perimeter=True, aspect_ratio=True, solidity=True, feret_diameter=True,
    saa=True, char_gb=True, make_skim_prop=True, throw=False, append=False)
gstslice.find_neigh()
print("slice", gstslice.uid if hasattr(gstslice, "uid") else "",
      "grains", len(np.unique(gstslice.lgi)) - 1)

## 2. Pixel grid → geometric polycrystal

`polygonised_grain_structure.pix_to_geom` builds per-grain Shapely polygons and the GB network used by smoothing and meshing.

In [ ]:
geom = polygonised_grain_structure(gstslice.lgi, gstslice.gid, gstslice.neigh_gid)
geom.pix_to_geom(verbose=False)
fig, ax = geom.plotgs(geom.POLYXTAL, figsize=(5, 5), dpi=110)
ax.set_title("raw polygons")
fig

## 3. Smooth grain-boundary segments

`npasses` is the number of smoothing passes. `min_segment_length_factor` skips very short GB segments.

In [ ]:
npasses = 2
min_segment_length_factor = 2
gsname = f"gs.{npasses}passes.{min_segment_length_factor}minseglenfactor"
geom.smooth_gbsegs(
    geom.GB, npasses=npasses,
    max_smooth_levels=np.repeat(min_segment_length_factor, npasses),
    plot=False, name=gsname)
pxtal = geom.smoothed[gsname]["POLYXTAL"]
fig, ax = geom.plotgs(pxtal, cmap="tab20", figsize=(5, 5), dpi=110)
ax.set_axis_off()
ax.set_title("smoothed polygons")
fig

## 4. Handoff to conformal meshing

`mesh_gs` consumes `{grain_id: Polygon | MultiPolygon}`. Plot and Abaqus export are library calls, not notebook-local INP writers.

In [ ]:
cells = {i + 1: poly for i, poly in enumerate(pxtal.geoms)}
result = mesh_gs(
    cells, mesh_size_gb=1.0, mesh_size_bulk=2.0,
    mesh_algo=6, recombine_to_quads=False, verbose=True)
m = result["mesher"]
m.form_elsets_gmsh()
m.build_boundary_nsets()
m.build_gb_nset()
print(m.validation_report)
fig, ax = visualize_gs_mesh(result, figsize=(6, 6), dpi=120, show_nsets=True)
fig

In [ ]:
out = Path.cwd() / "confMesh2d_geometrify_out"
inp = m.export_abaqus_inp(out / "rve_cps3.inp", plane="stress")
inp, summarize_inp(inp)